# Preflight Check
### Purpose: Download the active dataset from `config.yaml`, then verify the environment and pipeline

For **public** mode, run `00_build_public_sample.ipynb` first (same folder) so NEON crowns are converted into Solafune-shaped files under `data/public_sample/`.


In [1]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()
_start = Path(os.getenv("PROJECT_ROOT") or Path.cwd()).resolve()
root = next(p for p in [_start, *_start.parents] if (p / "config.yaml").exists())
sys.path[:0] = [str(root), str(root / "src")]

from src.utils.config import Config
from src.utils.dataset_source import download_dataset
from src.utils.helpers import c, init_notebook, p, simple_estimate_runtime, t
from src.utils.preflight import (
    check_config, check_data, check_dataset, check_disk_space, check_gpu, check_model,
)

config = Config.load(root=root)
init_notebook(config.train.seed)



=== init_notebook ===
Done


In [2]:
# Pull files for dataset.active in config.yaml (NEON zip when public).
# For public mode, run 00_build_public_sample.ipynb after this if annotations are missing.
download_summary = download_dataset(config, force=False)

from pathlib import Path

from src.utils.dataset_source import get_active_source

_src = get_active_source(config)
if _src["key"] == "public":
    _ann = Path(config.paths.annotations) if config.paths.annotations else None
    if _ann is None or not _ann.exists():
        p(
            "Public sample not built",
            "Run 00_build_public_sample.ipynb next (this folder)",
            color1=c.ORANGE,
        )
    else:
        p("Public sample", f"annotations at {_ann.name}", color1=c.GREEN)


=== Dataset download ===
Active source: public — NeonTreeEvaluation crown subset (Solafune-shaped)
Catalog: https://zenodo.org/records/15354422
Download dir: /Volumes/Colibri/Projects/cap6415-tree-canopy-detection/data/public_sample/raw
Notes: Public individual-tree crowns (~11 MB). Converted by notebook 00 into Solafune JSON under data/public_sample/. Not a drop-in for group_of_trees.
skip (exists): data.zip
Done: downloaded=0 skipped=1 manual=0
Public sample: annotations at train_annotations.json


In [3]:
t("PRE-FLIGHT CHECK")


=== PRE-FLIGHT CHECK ===


In [4]:
import traceback

checks = [
    ("Configuration", lambda: check_config(config)),
    ("GPU", lambda: check_gpu()),
    ("Data", lambda: check_data(config)),
    ("Dataset", lambda: check_dataset(config)),
    ("Model", lambda: check_model()),
    ("Disk Space", lambda: check_disk_space(config)),
]

results = {}
for name, func in checks:
    try:
        results[name] = func()
        p("")
    except Exception as e:
        results[name] = False
        p(f"✗ {name} check crashed", str(e), color1=c.RED)
        traceback.print_exc()


=== Checking Configuration ===
✓ Config loaded: /Volumes/Colibri/Projects/cap6415-tree-canopy-detection
✓ annotations: exists
✓ train_images: exists
✓ eval_images: exists
✓ notebooks: exists
✓ models: exists
✓ plots: exists

=== Checking GPU ===
✓ MPS available: Apple GPU

=== Checking Data ===
✓ Annotations loaded: 14 images
✓ Sample image: found
✓ Image loading: shape=(400, 400, 3)

=== Checking Dataset ===
✓ Dataset created: 5 samples
✓ Image shape: 3 items
  0: 3
  1: 128
  2: 128
✓ Mask shape: 2 items
  0: 128
  1: 128
✓ Image format: correct [3, H, W]
✓ Mask format: correct [H, W] for multi-class
✓ Mask values: valid classes: [0, 1]

=== Checking Model ===
✓ Model created: simple_cnn
Model parameters: 38,211
✓ Forward pass: output shape=torch.Size([2, 3, 256, 256])
✓ Output shape: correct (multi-class)

=== Checking Disk Space ===
Free space: 885.4 GB
✓ Sufficient space: >10 GB available



In [5]:
try:
    estimate = simple_estimate_runtime(config)
    p("Runtime estimate", estimate)
except Exception as e:
    p("Runtime estimate skipped", str(e), color1=c.ORANGE)


=== Runtime Estimate ===
Device: MPS (Apple GPU)
Training samples: 11
Batch size: 2
Image size: 128
Batches per epoch: 6
Epochs: 5
Est. sec / batch: 0.38
Estimated time per epoch: ~00m 02s
Estimated total time: ~00m 11s
Runtime estimate


In [6]:
t("SUMMARY")
passed = sum(1 for v in results.values() if v)
p("Passed", f"{passed}/{len(results)}")
for name, ok in results.items():
    if name == "GPU" and not ok:
        p(name, "FAIL — OK, defaulting to CPU", color1=c.ORANGE)
    else:
        p(name, "OK" if ok else "FAIL", color1=c.GREEN if ok else c.RED)


=== SUMMARY ===
Passed: 6/6
Configuration: OK
GPU: OK
Data: OK
Dataset: OK
Model: OK
Disk Space: OK
